In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install -U huggingface_hub

In [ ]:
# !pip install git+https://github.com/intel/auto-round.git --break-system-packages

In [1]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor
from safetensors import safe_open
import safetensors.torch as st
import gc
import json

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


PyTorch Version: 2.10.0+cu128
CUDA Available: True
CUDA Version: 12.8
GPU Name: NVIDIA L40
VRAM: 44.4 GB


In [4]:
notebook_login()

In [5]:
MODEL_ID = "google/gemma-4-12B-it"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound-12B"
LOCAL_PATH = "./local_model-12B"

In [6]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

A new version of huggingface_hub (1.17.0) is available! You are using version 1.7.1.
To update, run: pip install -U huggingface_hub

Fetching 9 files: 100%|███████████████████████████| 9/9 [03:05<00:00, 20.57s/it]
Download complete: : 24.0GB [03:05, 237MB/s]              /workspace/local_model-12B
Download complete: : 24.0GB [03:05, 129MB/s]


In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, 
    dtype=torch.bfloat16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

In [8]:
model

Gemma4UnifiedForConditionalGeneration(
  (model): Gemma4UnifiedModel(
    (language_model): Gemma4UnifiedTextModel(
      (embed_tokens): Gemma4UnifiedTextScaledWordEmbedding(262144, 3840, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4UnifiedTextDecoderLayer(
          (self_attn): Gemma4UnifiedTextAttention(
            (q_proj): Linear(in_features=3840, out_features=4096, bias=False)
            (q_norm): Gemma4UnifiedRMSNorm()
            (k_norm): Gemma4UnifiedRMSNorm()
            (v_norm): Gemma4UnifiedRMSNorm()
            (k_proj): Linear(in_features=3840, out_features=2048, bias=False)
            (v_proj): Linear(in_features=3840, out_features=2048, bias=False)
            (o_proj): Linear(in_features=4096, out_features=3840, bias=False)
          )
          (mlp): Gemma4UnifiedTextMLP(
            (gate_proj): Linear(in_features=3840, out_features=15360, bias=False)
            (up_proj): Linear(in_features=3840, out_features=15360, bias=False)
       

In [9]:
layer_config = {}
for name, _ in model.named_modules():
    if name.startswith(("model.vision_tower", "model.audio_tower",
                        "model.multi_modal_projector", "model.audio_projector")):
        layer_config[name] = {"bits": 32}

print(f"Skipping {len(layer_config)} non-LM modules")


Skipping 0 non-LM modules


In [10]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 0,               # RTN mode — required for Gemma 4
    "disable_opt_rtn": True,
    "nsamples": 256,
    "seqlen": 2048,
    "low_gpu_mem_usage": False,
    "quant_nontext_module": False,
    "layer_config": layer_config,
}

In [11]:
def push_to_hub(local_dir, repo_name, token):
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")
    try:
        api = HfApi()
        create_repo(full_repo_id, repo_type="model", exist_ok=True, private=False, token=token)
        api.upload_folder(folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token)
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [12]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-06-04 13:06:32 INFO entry.py L587: Using MLLM mode for multimodal model.


In [13]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round,auto_gptq,auto_awq", inplace=True
)

2026-06-04 13:06:34 WARNING logging.py L340: some layers are skipped quantization (shape not divisible by 32): 
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
2026-06-04 13:06:35 INFO zero_shot.py L135: RTN mode detected (iters=0): force blockwise quantization to avoid layer-wise full-model materialization.
Quantizing model.language_model.layers.0:   0%|          | 0/48 [00:00<?, ?it/s]2026-06-04 13:06:35 INFO offload.py L706: OffloadManager (compressor): tempdir = /workspace/ar_work_space/offload/compressor_97ydqb9u
2026-06-04 13:06:36 INFO device.py L1840: 'peak_ram': 1.49GB, 'peak_vram': 22.71GB
packing: 100%|██████████| 328/328 [00:23<00:00, 13.83it/s]


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

packing: 100%|██████████| 328/328 [00:30<00:00, 10.79it/s]


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-04 13:09:03 INFO export.py L166: Saving quantized model to auto_awq format
packing: 100%|██████████| 328/328 [00:24<00:00, 13.28it/s]


Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-04 13:09:51 INFO device.py L1840: 'peak_ram': 1.65GB, 'peak_vram': 22.71GB


(Gemma4UnifiedForConditionalGeneration(
   (model): Gemma4UnifiedModel(
     (language_model): Gemma4UnifiedTextModel(
       (embed_tokens): Gemma4UnifiedTextScaledWordEmbedding(262144, 3840, padding_idx=0)
       (layers): ModuleList(
         (0-4): 5 x Gemma4UnifiedTextDecoderLayer(
           (self_attn): Gemma4UnifiedTextAttention(
             (q_proj): WQLinear_GEMM(in_features=3840, out_features=4096, bias=False, w_bit=4, group_size=128)
             (q_norm): Gemma4UnifiedRMSNorm()
             (k_norm): Gemma4UnifiedRMSNorm()
             (v_norm): Gemma4UnifiedRMSNorm()
             (k_proj): WQLinear_GEMM(in_features=3840, out_features=2048, bias=False, w_bit=4, group_size=128)
             (v_proj): WQLinear_GEMM(in_features=3840, out_features=2048, bias=False, w_bit=4, group_size=128)
             (o_proj): WQLinear_GEMM(in_features=4096, out_features=3840, bias=False, w_bit=4, group_size=128)
           )
           (mlp): Gemma4UnifiedTextMLP(
             (gate_proj):

In [14]:
def fix_softcap_tensors(model_dir):
    print(f"\n[Fix] Scanning {model_dir}...")
    for fname in sorted(os.listdir(model_dir)):
        if not fname.endswith(".safetensors"):
            continue
        fpath = os.path.join(model_dir, fname)
        with safe_open(fpath, framework="pt", device="cpu") as f:
            all_keys = list(f.keys())
            bad_keys = [k for k in all_keys if "softcap" in k]
            if not bad_keys:
                print(f"  {fname}: clean")
                continue
            good_tensors = {k: f.get_tensor(k) for k in all_keys if k not in bad_keys}
        st.save_file(good_tensors, fpath)
        del good_tensors
        gc.collect()
        print(f"  {fname}: removed {len(bad_keys)} softcap entries → {bad_keys}")

    idx_path = os.path.join(model_dir, "model.safetensors.index.json")
    if os.path.exists(idx_path):
        with open(idx_path) as f:
            idx = json.load(f)
        bad_idx = [k for k in list(idx["weight_map"].keys()) if "softcap" in k]
        for k in bad_idx:
            del idx["weight_map"][k]
        with open(idx_path, "w") as f:
            json.dump(idx, f, indent=2)
        print(f"  index.json: removed {len(bad_idx)} entries")
    print("[Fix] ✅ Done")



In [16]:
fix_softcap_tensors(os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-round-auto-gptq"))
fix_softcap_tensors(os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-gptq"))
fix_softcap_tensors(os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-awq"))


[Fix] Scanning ./AutoRound-12B/local_model-12B-w4g128/auto-round-auto-gptq...
  model-00001-of-00002.safetensors: clean
  model-00002-of-00002.safetensors: clean
  index.json: removed 0 entries
[Fix] ✅ Done

[Fix] Scanning ./AutoRound-12B/local_model-12B-w4g128/auto-gptq...
  model-00001-of-00002.safetensors: clean
  model-00002-of-00002.safetensors: clean
  index.json: removed 0 entries
[Fix] ✅ Done

[Fix] Scanning ./AutoRound-12B/local_model-12B-w4g128/auto-awq...
  model-00001-of-00002.safetensors: clean
  model-00002-of-00002.safetensors: clean
  index.json: removed 0 entries
[Fix] ✅ Done


In [17]:
def copy_processor_configs(model_id, output_dir):
    for filename in ["preprocessor_config.json", "processor_config.json", "chat_template.jinja"]:
        try:
            shutil.copy(hf_hub_download(model_id, filename), output_dir)
            print(f"  Copied {filename} → {output_dir}")
        except Exception:
            pass

In [18]:
copy_processor_configs(MODEL_ID, os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-round-auto-gptq"))
copy_processor_configs(MODEL_ID, os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-gptq"))
copy_processor_configs(MODEL_ID, os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-awq"))

In [19]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [21]:
if hf_token:
    # Push auto_round format
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-round-auto-gptq"),
        f"{base_name}-W4A16-AutoRound",
        hf_token
    )
    # Push auto_gptq format (vLLM compatible)
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-gptq"),
        f"{base_name}-W4A16-AutoRound-GPTQ",
        hf_token
    )
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-12B-w4g128/auto-awq"),
        f"{base_name}-W4A16-AutoRound-AWQ",
        hf_token
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")



[Hub] Pushing ./AutoRound-12B/local_model-12B-w4g128/auto-round-auto-gptq to Vishva007/gemma-4-12B-it-W4A16-AutoRound...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-12B-it-W4A16-AutoRound

[Hub] Pushing ./AutoRound-12B/local_model-12B-w4g128/auto-gptq to Vishva007/gemma-4-12B-it-W4A16-AutoRound-GPTQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-12B-it-W4A16-AutoRound-GPTQ

[Hub] Pushing ./AutoRound-12B/local_model-12B-w4g128/auto-awq to Vishva007/gemma-4-12B-it-W4A16-AutoRound-AWQ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

[Hub] ✅ Successfully uploaded: https://huggingface.co/Vishva007/gemma-4-12B-it-W4A16-AutoRound-AWQ
